# NBT modelling summary

This notebook compares the winning data treatment, feature configuration and algorithm across the three prespecified prediction targets. Model selection occurred in development cross-validation; the values below are from the frozen untouched test sets.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULT_DIR = PROJECT_ROOT / "result" / "modeling"
winner_summary = pd.read_csv(RESULT_DIR / "cross_target_winner_summary.csv")
winner_summary

,target,winning missing strategy,winning feature configuration,winning model,primary metric,primary value,secondary metric,secondary value
0,duration_error_mins,"Missing-aware, priority retained",Both procedure levels,XGBoost,MAE,31.017171,R2,0.439344
1,operation_length_mins,"Missing-aware, priority retained",Both procedure levels,XGBoost,MAE,30.452553,R2,0.711676
2,meaningful_overrun_flag,"Missing-aware, priority retained",Full approved,XGBoost,PR-AUC,0.794674,Recall,0.763074


## Winning type of data

The winner table states whether priority was retained through the selected missing strategy, which feature representation won, and whether the neural network or another algorithm performed best. Start hour, flagged-record exclusion and complete cases remain sensitivity analyses rather than primary data choices.

In [2]:
importance = pd.read_csv(RESULT_DIR / "cross_target_feature_importance.csv")
top_features = (
    importance.sort_values(["target", "rank"])
    .groupby("target", as_index=False, group_keys=False)
    .head(10)
)
top_features

,target,feature,importance mean,importance SD,rank
0,duration_error_mins,ExpectedDurationMins,14.678087,0.321107,1
1,duration_error_mins,anaesthetic_desc,4.219187,0.268199,2
2,duration_error_mins,procedure_code_group,3.907621,0.190496,3
3,duration_error_mins,procedure_code_category,3.702649,0.266029,4
4,duration_error_mins,intended_management_label,3.134275,0.172161,5
5,duration_error_mins,admission_type_label,2.332894,0.190750,6
6,duration_error_mins,ASAScore,0.690551,0.120019,7
7,duration_error_mins,age_at_operation,0.452073,0.134055,8
8,duration_error_mins,sex_national_code,0.184665,0.068723,9
9,duration_error_mins,priority_level_label,-0.106241,0.078906,10


## Interpretation

Permutation importance measures predictive contribution on the frozen test population and is not a causal effect. Correlated features can share importance. A low-ranked feature may still be clinically important, and external validation remains necessary before deployment.